# Evidence Recall and Context-Length Analysis

This notebook analyzes **actual** outputs produced by
`scripts/evaluate_model.py`. It does not create placeholder metric values.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.context_length_analysis import (
    aggregate_context_length_metrics,
    save_context_analysis,
)
from src.model_evaluation import write_manual_error_analysis
from src.visualization import plot_context_length_analysis


## 1. Load evaluation results

Run `python scripts/evaluate_model.py` first.


In [ ]:
results_path = PROJECT_ROOT / "outputs" / "qa_examples.csv"
results = pd.read_csv(results_path)
results.head()


## 2. Evidence-recall audit table


In [ ]:
audit_columns = [
    "question",
    "predicted_answer",
    "reference_answer",
    "predicted_evidence",
    "reference_evidence",
    "exact_match",
    "token_f1",
    "evidence_recall",
    "observation",
]
results[audit_columns]


## 3. Performance by context length


In [ ]:
context_summary = aggregate_context_length_metrics(results)
context_summary


In [ ]:
save_context_analysis(context_summary, PROJECT_ROOT / "outputs")
plot_context_length_analysis(
    context_summary,
    PROJECT_ROOT / "outputs" / "context_length_analysis.png",
)


## 4. Inspect weak and failed cases


In [ ]:
weak_cases = results.sort_values(
    ["evidence_recall", "token_f1", "confidence_proxy"],
    ascending=[True, True, True],
)
weak_cases[audit_columns + ["warnings", "error"]].head(12)


## 5. Generate manual error-analysis Markdown


In [ ]:
write_manual_error_analysis(
    results,
    PROJECT_ROOT / "outputs" / "manual_error_analysis.md",
)


## Interpretation checklist

Review whether errors are caused by:

- a missing answer in the document;
- an ambiguous question;
- a wrong start or end token;
- a similar answer in another paragraph;
- a token-window boundary;
- PDF extraction noise;
- domain shift from SQuAD;
- a misleadingly high or low confidence proxy.
